In [1]:
import os
import re
import json
import html
import asyncio
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel
from IPython.display import display, Markdown, HTML
import gradio as gr
from openai import AsyncOpenAI

# OpenAI Agents SDK
from agents import (
    Agent,
    Runner,
    trace,
    handoff,
    set_default_openai_client,
    set_tracing_export_api_key,
)
from agents.mcp import MCPServerStdio
from contextlib import AsyncExitStack

load_dotenv(override=True)


def _ensure_chrys_on_path() -> None:
    """So `import agent_team_workspace` works from chrys/, agents/, or 6_mcp/."""
    import sys

    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "6_mcp/community_contributions/chrys",
        cwd / "community_contributions/chrys",
    ]
    for d in candidates:
        if (d / "agent_team_workspace.py").is_file():
            s = str(d)
            if s not in sys.path:
                sys.path.insert(0, s)
            return


_ensure_chrys_on_path()

print("✅ Setup complete. OpenRouter + OpenAI Agents SDK ready.")

✅ Setup complete. OpenRouter + OpenAI Agents SDK ready.


In [2]:
from typing import List, Optional

class AgentSpec(BaseModel):
    name: str
    role: str
    instructions: str
    handoff_description: str = ""
    tools: List[str] = []
    mcp_servers: List[dict] = []

class TeamSpec(BaseModel):
    manager: AgentSpec
    specialists: List[AgentSpec]
    success_criteria: List[str]
    domain: str = "general"
    generated_dir: Optional[str] = None
    primary_language: Optional[str] = None

In [3]:

def sanitize_name(name: str) -> str:
    return "".join(c if c.isalnum() else "_" for c in (name or "agent").lower())


_CHRYS = Path.cwd().resolve()
if _CHRYS.name == "chrys" and _CHRYS.parent.name == "community_contributions":
    WEEK6_MCP_ROOT = _CHRYS.parent.parent
else:
    WEEK6_MCP_ROOT = Path(os.getenv("WEEK6_MCP_ROOT", ".")).resolve()

# Shown to the team generator so it fills `tools` per role
MCP_TOOL_IDS_FOR_LLM = """
Each agent has a `tools` list (string ids). Pick tools that match the role. Valid ids:

- fetch — read/fetch web pages (uvx mcp-server-fetch; no API key)
- duckduckgo — DuckDuckGo search (`uvx` from nickclyde GitHub; clean stdio JSON-RPC)
- playwright — browser automation / page interaction (npx @playwright/mcp; run `npx playwright install` if needed)
- serper — Google search via Serper (`uvx serper-mcp-server`; SERPER_API_KEY or SERPAPI_API_KEY)
- memory — persistent scratchpad per agent (local libsql file under the project)
- pushover — push notifications via course push_server (PUSHOVER_USER, PUSHOVER_TOKEN; runs from 6_mcp)
- resend — send email via Resend (RESEND_API_KEY; npx resend-mcp)
- alphavantage — market data (ALPHAVANTAGE_API_KEY; npx alphavantage-stock-mcp)

Give the manager coordination + notify tools (pushover, resend). Give researchers fetch+duckduckgo+playwright+serper+memory.
Give trading/finance roles alphavantage + duckduckgo. For sales, marketing, GTM, or customer-facing teams, add a
Communications (or RevOps) specialist with resend (and optionally pushover): they write outbound email
copy as semantic HTML (headings, lists, tables, inline CSS safe for email clients) when using the Resend MCP.
You may assign 3–6 tools per specialist.
"""


def build_stdio_params(
    tool_id: str,
    agent_slug: str,
    *,
    week6: Path,
    memory_parent: Path | None = None,
) -> dict | None:
    """Return MCPServerStdio param dict, or None if tool unavailable (missing creds, etc.)."""
    w = week6
    tid = (tool_id or "").strip().lower().replace(" ", "_").replace("-", "_")
    if tid == "fetch":
        return {"command": "uvx", "args": ["mcp-server-fetch"]}
    if tid in ("duckduckgo", "duckduckgo_search", "ddg_search"):
        return {
            "command": "uvx",
            "args": [
                "--from",
                "git+https://github.com/nickclyde/duckduckgo-mcp-server",
                "duckduckgo-mcp-server",
            ],
        }
    if tid in ("playwright", "browser"):
        return {
            "command": "npx",
            "args": ["-y", "@playwright/mcp@latest"],
        }
    if tid == "serper" and (os.getenv("SERPER_API_KEY") or os.getenv("SERPAPI_API_KEY")):
        key = os.getenv("SERPER_API_KEY") or os.getenv("SERPAPI_API_KEY") or ""
        return {
            "command": "uvx",
            "args": ["serper-mcp-server"],
            "env": {"SERPER_API_KEY": key},
        }
    if tid == "memory":
        root = memory_parent or (Path.cwd() / "mcp_memory_dbs")
        root.mkdir(parents=True, exist_ok=True)
        db_path = (root / f"{agent_slug}.db").resolve()
        return {
            "command": "npx",
            "args": ["-y", "mcp-memory-libsql"],
            "env": {"LIBSQL_URL": f"file:{db_path}"},
        }
    if tid == "pushover" and os.getenv("PUSHOVER_USER") and os.getenv("PUSHOVER_TOKEN"):
        return {"command": "uv", "args": ["run", "push_server.py"], "cwd": str(w)}
    if tid == "resend" and os.getenv("RESEND_API_KEY"):
        return {
            "command": "npx",
            "args": ["-y", "resend-mcp"],
            "env": {"RESEND_API_KEY": os.getenv("RESEND_API_KEY", "")},
        }
    if tid == "alphavantage" and os.getenv("ALPHAVANTAGE_API_KEY"):
        return {
            "command": "npx",
            "args": ["-y", "alphavantage-stock-mcp"],
            "env": {"ALPHAVANTAGE_API_KEY": os.getenv("ALPHAVANTAGE_API_KEY", "")},
        }
    return None


def effective_tool_ids(
    spec: AgentSpec, *, week6: Path, memory_parent: Path | None = None
) -> list[str]:
    """Model-listed tools first, then add any extra tools whose credentials are available."""
    seen: list[str] = []
    for t in spec.tools or []:
        x = (t or "").strip().lower().replace(" ", "_").replace("-", "_")
        if x and x not in seen:
            seen.append(x)
    for extra in (
        "fetch",
        "memory",
        "pushover",
        "resend",
        "duckduckgo",
        "playwright",
        "serper",
        "alphavantage",
    ):
        if extra not in seen and build_stdio_params(
            extra,
            sanitize_name(spec.name),
            week6=week6,
            memory_parent=memory_parent,
        ):
            seen.append(extra)
    return seen


def all_team_tool_ids(
    team: TeamSpec, *, week6: Path, memory_parent: Path | None = None
) -> list[str]:
    u: set[str] = set()
    for spec in [team.manager, *team.specialists]:
        for t in effective_tool_ids(spec, week6=week6, memory_parent=memory_parent):
            u.add(t)
    return sorted(u)


def _mem_key(slug: str) -> tuple[str, str]:
    return ("memory", slug)


async def connect_team_mcp_servers(
    team: TeamSpec,
    stack: AsyncExitStack,
    *,
    week6: Path,
    memory_parent: Path,
):
    servers: dict[str | tuple[str, str], object] = {}
    for tid in all_team_tool_ids(team, week6=week6, memory_parent=memory_parent):
        if tid == "memory":
            continue
        p = build_stdio_params(tid, "na", week6=week6, memory_parent=memory_parent)
        if p:
            try:
                servers[tid] = await stack.enter_async_context(
                    MCPServerStdio(
                        params=p,
                        client_session_timeout_seconds=120,
                        name=f"mcp_{tid}",
                    )
                )
            except Exception as e:
                print(f"⚠️ Skipping MCP tool {tid!r} (connection failed): {e}")
    for spec in [team.manager, *team.specialists]:
        if "memory" not in effective_tool_ids(
            spec, week6=week6, memory_parent=memory_parent
        ):
            continue
        slug = sanitize_name(spec.name)
        k = _mem_key(slug)
        if k in servers:
            continue
        p = build_stdio_params("memory", slug, week6=week6, memory_parent=memory_parent)
        if p:
            try:
                servers[k] = await stack.enter_async_context(
                    MCPServerStdio(
                        params=p,
                        client_session_timeout_seconds=120,
                        name=f"mcp_memory_{slug}",
                    )
                )
            except Exception as e:
                print(
                    f"⚠️ Skipping memory MCP for {slug!r} (connection failed): {e}"
                )
    return servers


def mcps_for_spec(
    spec: AgentSpec,
    servers: dict,
    *,
    week6: Path,
    memory_parent: Path,
) -> list:
    out = []
    for t in effective_tool_ids(spec, week6=week6, memory_parent=memory_parent):
        if t == "memory":
            k = _mem_key(sanitize_name(spec.name))
            if k in servers:
                out.append(servers[k])
        elif t in servers:
            out.append(servers[t])
    return out


In [4]:

# OpenRouter configuration - uses your API key
openrouter_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:7860",
        "X-Title": "Agent Generator",
    }
)
set_default_openai_client(openrouter_client)

set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))

# Team JSON via chat.completions JSON mode (not AgentOutputSchema) — some OpenRouter models emit
# completionState trees or double-encoded strings; agent_team_workspace.coerce_team_dict_for_teamspec repairs that.
TEAM_GENERATOR_MODEL = os.getenv("TEAM_GENERATOR_MODEL", "openai/gpt-oss-120b")

TEAM_GENERATOR_SYSTEM = (
    """You are an expert at building high-performing teams of AI agents.
Given a prompt, create a complete team with a manager and specialists.
Focus heavily on software engineering when the prompt mentions engineering/manager/dev/team.

Success criteria (field `success_criteria`): Write 4–7 bullets that can be verified from a SINGLE assistant
turn (the team's final written answer + tool use). Good examples: includes runnable code snippets or a
minimal module; lists concrete test cases or assertions as text; names files/packages and integration steps;
summarizes research with citations from tools. BAD examples for this lab: "100% test coverage", "deployed
prototype in production", "working CI/CD pipeline green", "stakeholder sign-off" — those require real
infrastructure; if the user wants delivery, phrase criteria as what the answer must CONTAIN (e.g. "Provides
a complete GitHub Actions workflow YAML and pytest file contents in the response").

Give every agent instructions that push toward concrete artifacts in the reply: code blocks, commands,
config excerpts, and explicit test examples — not roadmap-only decks unless the user asked for planning only.
For non-engineering teams, artifacts can be email HTML, talk tracks, or briefs; include a specialist who
owns Resend and formats customer-facing email as valid HTML.

Set `primary_language` from the user's prompt (e.g. python, typescript, javascript, go, rust). Default python.
For software work, every agent's instructions must say: when emitting code files, use markdown fences whose
FIRST line inside the fence is `# file: src/...` or `# file: tests/...` (or `// file:` for JS/TS), paths
relative to the `project/` folder — so runs can extract them into a real tree with tests under `tests/`.

Fill each agent's `tools` list using these MCP tool ids (strings):
"""
    + MCP_TOOL_IDS_FOR_LLM
    + """

CRITICAL JSON shape (respond with one JSON object only — parsed with Pydantic):
- Required root keys: manager, specialists, success_criteria, domain. Optional: primary_language, generated_dir (omit or null).
- `manager` = ONE plain AgentSpec object. `specialists` = ARRAY of plain AgentSpec objects.
- NEVER use root key `agents`, or root `name` / `description` instead of manager/specialists.
- NEVER emit `completionState`, `entries`, `type`, or String/Object wrappers — only normal JSON.
- NEVER stringify agents as JSON text inside the array; each specialist must be a full object inline.
- Each agent's `tools` array: list each tool id at most once; typically 3–8 ids total — never repeat the same id in a loop.
- If you include `generated_dir`, use "" or omit it; the app always sets the real path after generation.
- Each AgentSpec: name, role, instructions, handoff_description (string, use "" if none), tools (string[]), mcp_servers (use []).

Example (structure only):
{
  "manager": {"name":"...","role":"...","instructions":"...","handoff_description":"","tools":["pushover","memory"],"mcp_servers":[]},
  "specialists": [
    {"name":"...","role":"...","instructions":"...","handoff_description":"","tools":["fetch","memory"],"mcp_servers":[]}
  ],
  "success_criteria": ["..."],
  "domain": "sales",
  "primary_language": "python",
  "generated_dir": ""
}

Return ONLY one JSON object matching that TeamSpec shape — no markdown fences or commentary."""
)


In [5]:
from agent_team_workspace import coerce_team_dict_for_teamspec, extract_json_object_text


async def generate_team(prompt: str) -> TeamSpec:
    """Generate team spec via OpenRouter chat.completions (json_object) + Pydantic."""
    print(f"🔄 Generating team for prompt: {prompt[:80]}...")

    try:
        with trace("team_generation"):
            resp = await openrouter_client.chat.completions.create(
                model=TEAM_GENERATOR_MODEL,
                messages=[
                    {"role": "system", "content": TEAM_GENERATOR_SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0.2,
            )
            raw = extract_json_object_text(resp.choices[0].message.content or "")
            data = json.loads(raw)
            data = coerce_team_dict_for_teamspec(data)
            team = TeamSpec.model_validate(data)

            slug = sanitize_name(team.manager.name or "team")
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            team.generated_dir = f"generated_teams/{slug}_{timestamp}"

            print(f"✅ Team generated successfully: {team.manager.name}")
            print(f"   Domain: {team.domain}")
            print(f"   Primary language: {team.primary_language or 'python'}")
            print(f"   Specialists: {len(team.specialists)}")
            return team

    except Exception as e:
        print(f"❌ Error generating team: {type(e).__name__}: {e}")
        raise


def export_generated_project(team: TeamSpec) -> None:
    import shutil

    from agent_team_workspace import ensure_project_layout

    _gd = (team.generated_dir or "").strip()
    if not _gd:
        raise ValueError("generated_dir is not set — generate the team first so the output folder exists.")
    base = Path(_gd)
    mem = base / "memory"
    mem.mkdir(parents=True, exist_ok=True)
    ensure_project_layout(
        generated_dir=str(base),
        manager_name=team.manager.name,
        primary_language=team.primary_language or "python",
    )
    week6 = WEEK6_MCP_ROOT
    tool_union = all_team_tool_ids(team, week6=week6, memory_parent=mem)

    (base / "PROJECT_README.md").write_text(
        f"""# Generated agent team

## Files
- `team_spec.json` — manager, specialists, tools, success criteria
- `team_runtime.py` — MCP wiring + `run_generated_team` (copy of course module)
- `run_team_manual.py` — run the team from the terminal
- `runs/` — **Run & Evaluate** writes `run_*.md`, `run_*_interactions.{{md,json}}`, `run_*.evaluation.json`, `latest_run.md`
- `project/` — language-specific **src/** and **tests/** tree; agents should tag code blocks with `# file: src/...` / `tests/...` for extraction
- `memory/` — per-agent libsql DBs for the `memory` MCP
- `project_manifest.json` — tool union + metadata
- `.env.example` — environment variables to copy

## Tools (union) for this team
{", ".join(tool_union)}

## Manual run
From this folder (`chrys/`), with repo `.venv` / `uv`:

```bash
uv run python {team.generated_dir}/run_team_manual.py "Your task for the team"
```

Set `WEEK6_MCP_ROOT` to the absolute path of `agents/6_mcp` if pushover (course `push_server.py`) does not start.

If the **Playwright** MCP fails to launch browsers, run once: `npx playwright install`.

## More MCPs in the wild
- [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers)
- [pulsemcp.com](https://www.pulsemcp.com/)
- [glama.ai/mcp/servers](https://glama.ai/mcp/servers)

The **notebook** loads `agent_team_workspace.py` (next to `agent_generator.ipynb`) to scaffold `project/` and to
record per-run interaction traces under `runs/` — keep that file alongside the notebook.

Add new stdio servers by extending `team_runtime.py` or regenerating teams with new tool ids in the LLM prompt.
""",
        encoding="utf-8",
    )

    (base / ".env.example").write_text(
        "\n".join(
            [
                "OPENROUTER_API_KEY=",
                "OPENAI_API_KEY=",
                "SERPER_API_KEY=",
                "SERPAPI_API_KEY=",
                "PUSHOVER_USER=",
                "PUSHOVER_TOKEN=",
                "RESEND_API_KEY=",
                "RESEND_TO_EMAIL=",
                "NOTIFY_EMAIL=",
                "RESEND_FROM_EMAIL=onboarding@resend.dev",
                "ALPHAVANTAGE_API_KEY=",
                "WEEK6_MCP_ROOT=",
                "",
            ]
        ),
        encoding="utf-8",
    )

    (base / "project_manifest.json").write_text(
        json.dumps(
            {
                "generated_dir": team.generated_dir,
                "domain": team.domain,
                "tool_union": tool_union,
                "success_criteria": team.success_criteria,
                "primary_language": team.primary_language or "python",
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    src = Path.cwd() / "team_project_runtime.py"
    if src.is_file():
        shutil.copy(src, base / "team_runtime.py")
    else:
        print("⚠️ team_project_runtime.py not found next to notebook; copy it manually into the project folder.")

    (base / "run_team_manual.py").write_text(
        '''#!/usr/bin/env python3
"""CLI runner for this generated team."""
import asyncio
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()


async def _main() -> None:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
    from team_runtime import run_generated_team

    here = Path(__file__).resolve().parent
    task = " ".join(sys.argv[1:]).strip() or "Briefly introduce the team and its roles."
    out = await run_generated_team(here, task)
    print(out)


if __name__ == "__main__":
    asyncio.run(_main())
''',
        encoding="utf-8",
    )


def persist_team(team: TeamSpec) -> str:
    """Write `team_spec.json` plus a runnable mini-project (README, CLI, runtime copy)."""
    _gd = (team.generated_dir or "").strip()
    if not _gd:
        raise ValueError("generated_dir is not set — run generate_team before persist_team.")
    base_dir = Path(_gd)
    base_dir.mkdir(parents=True, exist_ok=True)
    (base_dir / "memory").mkdir(parents=True, exist_ok=True)
    (base_dir / "team_spec.json").write_text(
        team.model_dump_json(indent=2), encoding="utf-8"
    )
    export_generated_project(team)
    print(f"✅ Project saved: {base_dir}")
    return str(base_dir)

In [6]:
def create_agent_for_spec(
    spec: AgentSpec,
    servers: dict,
    *,
    week6: Path,
    memory_parent: Path,
) -> Agent:
    return Agent(
        name=spec.name,
        instructions=spec.instructions,
        model="openai/openai/gpt-oss-120b",
        handoff_description=spec.handoff_description or f"Transfer to {spec.name}",
        mcp_servers=mcps_for_spec(
            spec, servers, week6=week6, memory_parent=memory_parent
        ),
        mcp_config={"convert_schemas_to_strict": False},
    )


async def build_team(
    team_spec: TeamSpec,
    servers: dict,
    *,
    week6: Path,
    memory_parent: Path,
):
    """Build manager with handoffs; each agent gets MCP tools from `servers`."""
    specialists = [
        create_agent_for_spec(s, servers, week6=week6, memory_parent=memory_parent)
        for s in team_spec.specialists
    ]
    mgr_instructions = team_spec.manager.instructions
    if not specialists:
        mgr_instructions += (
            "\n\nYou have no specialist agents to hand off to. Answer the user's task directly "
            "using your own expertise."
        )

    manager = Agent(
        name=team_spec.manager.name,
        instructions=mgr_instructions,
        handoffs=specialists,
        model="openai/openai/gpt-oss-120b",
        mcp_servers=mcps_for_spec(
            team_spec.manager,
            servers,
            week6=week6,
            memory_parent=memory_parent,
        ),
        mcp_config={"convert_schemas_to_strict": False},
    )

    return manager, specialists

In [7]:
def _output_has_substantial_code_fences(text: object, min_fence_chars: int = 120) -> bool:
    s = str(text or "")
    blocks = re.findall(r"```.*?```", s, flags=re.DOTALL)
    return sum(len(b) for b in blocks) >= min_fence_chars


from agent_team_workspace import (
    ensure_project_layout,
    extract_and_write_project_files,
    run_manager_streamed_collect,
    save_run_bundle,
)


def _markdownish_to_html_email(text: str) -> str:
    """Lightweight block → HTML for Resend (not full Markdown)."""
    chunks: list[str] = []
    for para in re.split("\n{2,}", str(text).strip()):
        p = para.strip()
        if not p:
            continue
        if p.startswith("```"):
            chunks.append(
                f'<pre style="white-space:pre-wrap;font-size:13px">{html.escape(p)}</pre>'
            )
        else:
            inner = html.escape(p).replace("\n", "<br/>")
            chunks.append(
                f'<p style="margin:0 0 12px;line-height:1.45">{inner}</p>'
            )
    if not chunks:
        chunks.append(
            f'<pre style="white-space:pre-wrap">{html.escape(str(text))}</pre>'
        )
    inner = "\n".join(chunks)
    return (
        "<!DOCTYPE html><html><body style=\"font-family:system-ui,sans-serif;max-width:720px\">"
        f"{inner}</body></html>"
    )


def _send_pushover(title: str, message: str) -> tuple[bool, str]:
    token, user = os.getenv("PUSHOVER_TOKEN"), os.getenv("PUSHOVER_USER")
    if not token or not user:
        return False, "PUSHOVER_TOKEN / PUSHOVER_USER not set"
    data = urlencode(
        {
            "token": token,
            "user": user,
            "title": title[:250],
            "message": message[:1024],
        }
    ).encode()
    req = Request("https://api.pushover.net/1/messages.json", data=data, method="POST")
    try:
        with urlopen(req, timeout=30) as resp:
            return True, resp.read().decode()
    except Exception as e:
        return False, str(e)


def _send_resend_html(subject: str, html_body: str, text_body: str) -> tuple[bool, str]:
    key = os.getenv("RESEND_API_KEY")
    to = os.getenv("RESEND_TO_EMAIL") or os.getenv("NOTIFY_EMAIL")
    from_addr = os.getenv("RESEND_FROM_EMAIL", "onboarding@resend.dev")
    if not key or not to:
        return False, "RESEND_API_KEY or RESEND_TO_EMAIL / NOTIFY_EMAIL not set"
    payload = json.dumps(
        {
            "from": from_addr,
            "to": [to],
            "subject": subject[:998],
            "html": html_body,
            "text": text_body[:50000],
        }
    ).encode()
    req = Request(
        "https://api.resend.com/emails",
        data=payload,
        method="POST",
        headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
    )
    try:
        with urlopen(req, timeout=60) as resp:
            return True, resp.read().decode()
    except Exception as e:
        return False, str(e)


async def maybe_notify_non_code_run(
    *,
    has_code_deliverables: bool,
    project_dir: Path,
    task: str,
    output: str,
    evaluation: dict,
) -> dict:
    """Sales / narrative runs: ping Pushover and/or email summary when there are no code fences."""
    out: dict = {"pushover": None, "resend": None, "note": None}
    if has_code_deliverables:
        out["note"] = "Substantial ``` code blocks in output — optional push/email skipped."
        return out
    preview = str(output)[:800] + ("…" if len(str(output)) > 800 else "")
    title = "Team run done (narrative / no code blocks)"
    msg = f"Saved: {project_dir / 'runs'}\n\nTask: {task[:220]}\n\n{preview}"
    if os.getenv("PUSHOVER_USER") and os.getenv("PUSHOVER_TOKEN"):
        ok, detail = await asyncio.to_thread(_send_pushover, title, msg)
        out["pushover"] = {"ok": ok, "detail": detail}
    if os.getenv("RESEND_API_KEY") and (
        os.getenv("RESEND_TO_EMAIL") or os.getenv("NOTIFY_EMAIL")
    ):
        email_plain = (
            f"## Task\n\n{task}\n\n## Output\n\n{output}\n\n"
            f"## Score: {evaluation.get('score')} — Passed: {evaluation.get('passed')}\n\n"
            f"## Feedback\n\n{evaluation.get('feedback', '')}"
        )
        html_body = _markdownish_to_html_email(email_plain)
        subj = f"[Agent team] Run — score {evaluation.get('score', 'n/a')}"
        ok, detail = await asyncio.to_thread(
            _send_resend_html,
            subj,
            html_body,
            f"Task:\n{task}\n\nOutput:\n{output[:20000]}",
        )
        out["resend"] = {"ok": ok, "detail": detail}
    return out


async def run_team_with_evaluation(team_spec: TeamSpec, task: str):
    """Run the team on a task (with MCP tools), evaluate, save under generated_dir/runs/, notify if no code."""
    _gd = (team_spec.generated_dir or "").strip()
    if not _gd:
        raise ValueError(
            "Team is missing generated_dir. Generate a team in the notebook (or set generated_dir in team_spec.json)."
        )
    week6 = WEEK6_MCP_ROOT
    memory_parent = Path(_gd) / "memory"
    memory_parent.mkdir(parents=True, exist_ok=True)

    async with AsyncExitStack() as stack:
        servers = await connect_team_mcp_servers(
            team_spec, stack, week6=week6, memory_parent=memory_parent
        )
        manager, _ = await build_team(
            team_spec, servers, week6=week6, memory_parent=memory_parent
        )

        criteria_lines = "\n".join(f"- {c}" for c in team_spec.success_criteria)
        run_task = f"""{task}

--- Grading context (embed these into your answer where possible) ---
Success criteria you will be scored on:
{criteria_lines}

Project layout: implementation files must land under the generated `project/` folder. For each file, use a
markdown code fence whose first line is `# file: src/...` or `# file: tests/...` (or `// file:` for TS/JS)
with a path relative to `project/`.

Execution rules for this run: Prefer concrete deliverables in the final answer (code, tests as text, shell
commands, CI YAML, OpenAPI sketches) — not only narrative plans. Criteria that imply production deployment,
real CI runs, or human sign-off count as MET if the response supplies complete, actionable artifacts (files,
configs, steps) that would achieve them in a repo, not if they are literally done in this chat."""

        ensure_project_layout(
            generated_dir=str(Path(_gd)),
            manager_name=team_spec.manager.name,
            primary_language=team_spec.primary_language or "python",
        )
        with trace("team_execution"):
            output, timeline, interactions_md = await run_manager_streamed_collect(
                manager, run_task
            )
    
    # Evaluation: criteria are often one-shot; judge substance not literal production completion
    eval_prompt = f"""You grade a single model turn (no real deploy/CI/stakeholders).

Task: {task}
Model output: {output}
Success criteria (list): {team_spec.success_criteria}

Rubric:
- Score 0–100 for how well the output addresses the TASK and satisfies each criterion IN THE SENSE OF
  "demonstrated in text/code in this response". Partial credit for thorough specs, working snippets, and
  copy-pasteable configs even when a criterion wording sounds like a live system requirement.
- Set passed to true if score >= 70 OR if the output is substantially implementation-oriented (real code +
  tests or CI as text) and only fails criteria that explicitly require external reality (e.g. "URL is live")
  that no chat turn could prove.
- In feedback, distinguish "missing from answer" vs "impossible in one chat turn"; do not fail solely because
  the team did not literally deploy or run CI.

Return only JSON: {{"score": <int>, "passed": <bool>, "feedback": "<string>"}}"""

    eval_result = await openrouter_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": eval_prompt}],
        response_format={"type": "json_object"}
    )
    
    eval_data = json.loads(eval_result.choices[0].message.content)
    project_dir = Path(_gd).resolve()
    project_dir.mkdir(parents=True, exist_ok=True)
    proj_root = project_dir / "project"
    proj_root.mkdir(parents=True, exist_ok=True)
    extracted = extract_and_write_project_files(proj_root, str(output))
    saved_paths = save_run_bundle(
        project_dir,
        task=task,
        output=str(output),
        evaluation=eval_data,
        interactions_md=interactions_md,
        timeline=timeline,
        extracted_files=extracted,
    )
    has_code = _output_has_substantial_code_fences(output)
    notifications = await maybe_notify_non_code_run(
        has_code_deliverables=has_code,
        project_dir=project_dir,
        task=task,
        output=str(output),
        evaluation=eval_data,
    )
    return {
        "output": output,
        "evaluation": eval_data,
        "success_criteria": team_spec.success_criteria,
        "trace_id": None,
        "saved_paths": saved_paths,
        "extracted_project_files": extracted,
        "agent_timeline": timeline,
        "notifications": notifications,
        "had_code_deliverables": has_code,
    }

In [8]:
def create_gradio_interface():
    with gr.Blocks(title="Agent Team Generator") as demo:
        gr.Markdown(
            "# 🤖 AI Agent Team Generator\n"
            "Generate a team from your prompt, then run a task and score it against success criteria.\n\n"
            "**Models:** use `openai/<openrouter-model-id>` in Agents SDK (e.g. `openai/openai/gpt-oss-120b` for model id `openai/gpt-oss-120b`) "
            "so calls go through your OpenRouter client."
        )

        last_team_state = gr.State(value=None)

        with gr.Tab("Generate Team"):
            prompt_input = gr.Textbox(
                label="Describe the team you want",
                placeholder="You are an engineering manager. Spawn a team of engineers to build a full-stack web application with user authentication.",
                lines=5,
                value="You are an engineering manager, spawn a team of engineers that would work on an engineering task.",
            )
            generate_btn = gr.Button("Generate Team", variant="primary")

            output_md = gr.Markdown()
            team_output = gr.JSON(label="Team JSON")

            async def generate(prompt: str):
                if not prompt or not str(prompt).strip():
                    return "**Error:** Please enter a prompt.", {"error": "Empty prompt"}, None

                try:
                    team = await generate_team(prompt)
                    persist_team(team)
                    dumped = team.model_dump()

                    md = f"""
**✅ Team generated**

**Manager:** {team.manager.name}
**Domain:** {team.domain}
**Primary language (project layout):** {team.primary_language or "python"}
**Specialists:** {len(team.specialists)}
**Saved project:** `{team.generated_dir or '(not set)'}/` — `team_spec.json`, `run_team_manual.py`, `team_runtime.py`, `PROJECT_README.md`, `memory/`

**Success criteria:**
""" + "\n".join([f"- {c}" for c in team.success_criteria])

                    return md, dumped, dumped
                except Exception as e:
                    import traceback

                    error_msg = f"**Error:** `{type(e).__name__}`: {e}"
                    print("Full traceback:")
                    traceback.print_exc()
                    return error_msg, {"error": str(e), "type": type(e).__name__}, None

            generate_btn.click(
                fn=generate,
                inputs=[prompt_input],
                outputs=[output_md, team_output, last_team_state],
            )

        with gr.Tab("Run & Evaluate"):
            gr.Markdown(
                "Uses the last generated team (manager + specialist handoffs). Each run writes "
"`runs/run_<ts>.md`, `run_<ts>_interactions.{md,json}`, `project/` file extraction, and `latest_run.md`. "
                "If the answer has **no** substantial ``` code blocks, optional **Pushover** and/or **Resend** "
                "summaries fire when `PUSHOVER_*` / `RESEND_API_KEY` + `RESEND_TO_EMAIL` (or `NOTIFY_EMAIL`) are set."
            )
            task_input = gr.Textbox(
                label="Task for the team",
                lines=4,
                placeholder="e.g. Implement a minimal JWT auth module: show Python code, 3 pytest cases as text, and a sample GitHub Actions workflow YAML in the answer.",
            )
            run_btn = gr.Button("Run task & evaluate", variant="primary")
            eval_output = gr.Markdown()

            async def run_eval(task: str, team_dict):
                if not team_dict:
                    return "## No team loaded\nGenerate a team in the **Generate Team** tab first."
                if not task or not str(task).strip():
                    return "## Enter a task\nDescribe what you want the team to produce."

                try:
                    team = TeamSpec.model_validate(team_dict)
                    result = await run_team_with_evaluation(team, str(task).strip())
                    ev = result.get("evaluation") or {}
                    out = result.get("output", "")
                    out_preview = str(out)[:6000]
                    sp = result.get("saved_paths") or {}
                    nt = result.get("notifications") or {}
                    code_flag = result.get("had_code_deliverables")
                    extracted = result.get("extracted_project_files") or []
                    ix_preview = ""
                    _ip = sp.get("interactions_md")
                    if _ip and Path(_ip).is_file():
                        try:
                            ix_preview = Path(_ip).read_text(encoding="utf-8")[:4000]
                        except OSError:
                            ix_preview = "(could not read interactions file)"
                    return f"""## Model output

{out_preview}

---

## Saved to generated project

- **Run log:** `{sp.get("markdown", "n/a")}`
- **Evaluation JSON:** `{sp.get("evaluation_json", "n/a")}`
- **Agent interactions (tools, handoffs, reasoning):** `{sp.get("interactions_md", "n/a")}`
- **Interactions (JSON):** `{sp.get("interactions_json", "n/a")}`
- **Latest copy:** `{sp.get("latest", "n/a")}`

**Files extracted into `project/`:** {extracted if extracted else "_(none — use `# file:` first line in code fences)_"}

### Interaction trace (preview)

```
{ix_preview}
```

**Had substantial code blocks (skips optional push/email):** {code_flag}

**Notifications** (Pushover / Resend when output is narrative-heavy):

```json
{json.dumps(nt, indent=2)}
```

---

## Evaluation vs success criteria

- **Score:** {ev.get("score", "n/a")}
- **Passed:** {ev.get("passed", "n/a")}
- **Feedback:** {ev.get("feedback", "n/a")}
"""
                except Exception as e:
                    import traceback

                    traceback.print_exc()
                    return f"**Error:** `{type(e).__name__}`: {e}"

            run_btn.click(
                fn=run_eval,
                inputs=[task_input, last_team_state],
                outputs=[eval_output],
            )

    return demo

# Launch with simpler settings
demo = create_gradio_interface()
demo.launch(
    inbrowser=True,
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


🔄 Generating team for prompt: You are an engineering manager, spawn a team of engineers that would work on an ...
✅ Team generated successfully: Alex Morgan
   Domain: software engineering
   Primary language: python
   Specialists: 4
✅ Project saved: generated_teams/alex_morgan_20260328_2157
